# Tenhou Data Scraping

This notebook walks through the scraping pipeline step by step.

**Key facts discovered:**
- Recent years (2025–2026): individual `scc{YEAR}{MMDD}.html.gz` files accessible at `https://tenhou.net/sc/raw/dat/`
- Older years (2012–2021): bundled in yearly zips at `https://tenhou.net/sc/raw/scraw{YEAR}.zip`
- `scc` prefix = 鳳凰卓 (Phoenix table) — highest ranked room
- Each html.gz lists games played that day with links to download the XML log

In [1]:
import gzip
import io
import os
import re
import struct
import zipfile

import requests
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)"}

## Step 1: Explore what's available

Fetch the file listing from Tenhou to see what scc (鳳凰卓) files exist.

In [2]:
# Recent files (last ~2 weeks)
r = requests.get("https://tenhou.net/sc/raw/list.cgi", headers=HEADERS)
recent_files = re.findall(r"file:'([^']+)'", r.text)
scc_recent = [f for f in recent_files if "scc" in f]
print(f"Recent scc files available: {len(scc_recent)}")
print(scc_recent[:5])

Recent scc files available: 201
['scc2026030500.html.gz', 'scc2026030501.html.gz', 'scc2026030502.html.gz', 'scc2026030503.html.gz', 'scc2026030504.html.gz']


In [3]:
# Older files (2025 and earlier) — listed at list.cgi?old
r = requests.get("https://tenhou.net/sc/raw/list.cgi?old", headers=HEADERS)
old_files = re.findall(r"file:'([^']+)'", r.text)
scc_old = [f for f in old_files if "scc" in f]
years_available = sorted(set(f.split("/")[0] for f in old_files))
print(f"Years available in old list: {years_available}")
print(f"scc files in old list: {len(scc_old)}")
# Note: years before 2025 are in yearly zips, not listed individually

Years available in old list: ['2025', '2026']
scc files in old list: 428


## Step 2: Fetch one html.gz file

For 2021, we need to work with the yearly zip. We use HTTP range requests to avoid downloading the full 513MB — we read only the zip's central directory (at the end of the file) to get file names and offsets, then fetch individual files.

In [ ]:
YEAR = 2020
ZIP_URL = f"https://tenhou.net/sc/raw/scraw{YEAR}.zip"

# Get total file size
r = requests.head(ZIP_URL, headers=HEADERS)
total_size = int(r.headers["Content-Length"])
print(f"scraw{YEAR}.zip size: {total_size / 1e6:.1f} MB")

scraw2021.zip size: 513.2 MB


In [5]:
def get_zip_central_directory(zip_url, total_size):
    """Fetch just the zip central directory using HTTP range requests."""
    # Read last 100KB to find the End of Central Directory record
    h = {**HEADERS, "Range": f"bytes={total_size - 102400}-{total_size - 1}"}
    tail = requests.get(zip_url, headers=h).content

    idx = tail.rfind(b"PK\x05\x06")  # EOCD signature
    eocd = tail[idx:]
    cd_size = struct.unpack("<I", eocd[12:16])[0]
    cd_offset = struct.unpack("<I", eocd[16:20])[0]

    # Fetch the central directory itself
    h = {**HEADERS, "Range": f"bytes={cd_offset}-{total_size - 1}"}
    cd_data = requests.get(zip_url, headers=h).content
    return cd_data, cd_offset


def parse_central_directory(cd_data):
    """Parse zip central directory entries into a dict of {filename: (local_header_offset, compressed_size)}."""
    entries = {}
    i = 0
    while i < len(cd_data):
        if cd_data[i:i+4] != b"PK\x01\x02":
            i += 1
            continue
        compressed_size = struct.unpack("<I", cd_data[i+20:i+24])[0]
        fname_len = struct.unpack("<H", cd_data[i+28:i+30])[0]
        extra_len = struct.unpack("<H", cd_data[i+30:i+32])[0]
        comment_len = struct.unpack("<H", cd_data[i+32:i+34])[0]
        local_header_offset = struct.unpack("<I", cd_data[i+42:i+46])[0]
        fname = cd_data[i+46:i+46+fname_len].decode("utf-8", errors="replace")
        entries[fname] = (local_header_offset, compressed_size)
        i += 46 + fname_len + extra_len + comment_len
    return entries


cd_data, cd_offset = get_zip_central_directory(ZIP_URL, total_size)
entries = parse_central_directory(cd_data)

scc_files = {k: v for k, v in entries.items() if "scc" in k}
print(f"Total files in zip: {len(entries)}")
print(f"scc (鳳凰卓) html.gz files: {len(scc_files)}")
print("Sample:", list(scc_files.keys())[:3])

Total files in zip: 1994
scc (鳳凰卓) html.gz files: 365
Sample: ['2021/scc20210101.html.gz', '2021/scc20210102.html.gz', '2021/scc20210103.html.gz']


In [6]:
def fetch_file_from_zip(zip_url, local_header_offset, compressed_size):
    """Fetch a single file from a zip using HTTP range requests."""
    # Local file header is 30 bytes + fname + extra (variable)
    # Fetch header first to find where data starts
    h = {**HEADERS, "Range": f"bytes={local_header_offset}-{local_header_offset + 512}"}
    header_bytes = requests.get(zip_url, headers=h).content
    fname_len = struct.unpack("<H", header_bytes[26:28])[0]
    extra_len = struct.unpack("<H", header_bytes[28:30])[0]
    data_start = local_header_offset + 30 + fname_len + extra_len

    # Fetch compressed data
    h = {**HEADERS, "Range": f"bytes={data_start}-{data_start + compressed_size - 1}"}
    compressed_data = requests.get(zip_url, headers=h).content

    # Decompress (deflate, wbits=-15 for raw deflate)
    import zlib
    return zlib.decompress(compressed_data, wbits=-15)


# Fetch the first scc file as a test
test_fname = sorted(scc_files.keys())[0]
local_offset, comp_size = scc_files[test_fname]
print(f"Fetching {test_fname} (compressed: {comp_size / 1024:.1f} KB)...")

raw_gz = fetch_file_from_zip(ZIP_URL, local_offset, comp_size)
html_content = gzip.decompress(raw_gz).decode("utf-8")
print(f"Decompressed html size: {len(html_content)} bytes")
print(html_content[:500])

Fetching 2021/scc20210101.html.gz (compressed: 36.1 KB)...
Decompressed html size: 149620 bytes
00:00 | 37 | 四鳳南喰赤－ | <a href="http://tenhou.net/0/?log=2021010100gm-00a9-0000-fc215cd8">牌譜</a> | kuro*(+40.0) 緑のランプ(+7.0) CHUGE(-18.0) ゆーFE(-29.0)<br>
00:01 | 26 | 四鳳南喰赤－ | <a href="http://tenhou.net/0/?log=2021010100gm-00a9-0000-c88844a8">牌譜</a> | もこみ(+45.0) yube(+5.0) sako＠最(-19.0) Comedian(-31.0)<br>
00:02 | 16 | 四鳳南喰赤－ | <a href="http://tenhou.net/0/?log=2021010100gm-00a9-0000-c531d3ac">牌譜</a> | 帰りたい。(+60.0) 滝本ひふみ！(+16.0) 愛と正義(-24.0) 玄化(-52.0)<br>
00:04 | 28 | 四鳳南喰赤－ | <a href="http://te


## Step 3: Parse with BeautifulSoup and filter for 鳳凰卓

In [7]:
def get_houou_links(html_content):
    """Extract game log links for 鳳凰卓 games using bs4."""
    soup = BeautifulSoup(html_content, "html.parser")
    links = []
    for a_tag in soup.find_all("a", href=True):
        href = a_tag["href"]
        if not href.startswith("http://tenhou.net/0/?log="):
            continue
        # Game type is in the same text line — check for 鳳 character
        line = a_tag.parent.get_text() if a_tag.parent else ""
        if "鳳" in line:
            links.append(href)
    return links


links = get_houou_links(html_content)
print(f"Found {len(links)} 鳳凰卓 games in {test_fname}")
print(links[:3])

Found 981 鳳凰卓 games in 2021/scc20210101.html.gz
['http://tenhou.net/0/?log=2021010100gm-00a9-0000-fc215cd8', 'http://tenhou.net/0/?log=2021010100gm-00a9-0000-c88844a8', 'http://tenhou.net/0/?log=2021010100gm-00a9-0000-c531d3ac']


## Step 4: Download one game log

In [8]:
def parse_batch_dir(fname, base_output_dir):
    """
    Parse date and batch number from scc filename.
    e.g. '2021/scc20210101.html.gz'    -> date='20210101', batch='00'
         '2026/scc2026031006.html.gz'  -> date='20260310', batch='06'
    Returns the batch directory path.
    """
    stem = os.path.basename(fname).replace("scc", "").replace(".html.gz", "")
    date = stem[:8]  # YYYYMMDD
    batch = stem[8:] if len(stem) > 8 else "00"
    batch_dir = os.path.join(base_output_dir, date, f"{date}_{batch}")

    return batch_dir


def download_game_log(href, batch_dir):
    """Download a single game log XML into the batch directory."""
    gid = href.split("=")[1]
    filepath = os.path.join(batch_dir, gid + ".log")

    if os.path.exists(filepath):
        return

    log_url = "https://tenhou.net/0/log/?" + gid
    response = requests.get(log_url, headers=HEADERS)
    if response.status_code == 200:
        with open(filepath, "wb") as f:
            f.write(response.content)
    else:
        print(f"  Failed: {log_url} -> {response.status_code}")


base_output_dir = f"processed/{YEAR}/"

# Test with the first scc file
test_batch_dir = parse_batch_dir(test_fname, base_output_dir)
print(f"Batch dir: {test_batch_dir}")

os.makedirs(test_batch_dir, exist_ok=True)
download_game_log(links[0], test_batch_dir)
print("Done")

Batch dir: processed/2021/20210101/20210101_00
Done


## Step 5: Scale to all scc files for the year

Once the above steps look good, run this to scrape all 鳳凰卓 games for the year.

In [9]:
from concurrent.futures import ThreadPoolExecutor, as_completed

MAX_WORKERS = 16  # number of parallel downloads — tune based on network/politeness

for fname in sorted(scc_files.keys()):
    batch_dir = parse_batch_dir(fname, base_output_dir)

    print(f"Processing {fname}...")
    try:
        local_offset, comp_size = scc_files[fname]
        raw_gz = fetch_file_from_zip(ZIP_URL, local_offset, comp_size)
        html = gzip.decompress(raw_gz).decode("utf-8")
        links = get_houou_links(html)

        # Skip if batch dir exists and file count matches number of games
        if os.path.exists(batch_dir):
            existing = len(os.listdir(batch_dir))
            if existing >= len(links):
                print(f"  Skipping — {existing}/{len(links)} files already present")
                continue
            else:
                print(f"  Incomplete — {existing}/{len(links)} files present, re-downloading")

        print(f"  {len(links)} 鳳凰卓 games found")
        os.makedirs(batch_dir, exist_ok=True)

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = {executor.submit(download_game_log, href, batch_dir): href for href in links}
            for future in as_completed(futures):
                exc = future.exception()
                if exc:
                    print(f"  Error downloading {futures[future]}: {exc}")

    except Exception as e:
        print(f"  Error: {e}")

Processing 2021/scc20210101.html.gz...
  Skipping — 981/981 files already present
Processing 2021/scc20210102.html.gz...
  Skipping — 1011/1011 files already present
Processing 2021/scc20210103.html.gz...
  Skipping — 947/947 files already present
Processing 2021/scc20210104.html.gz...
  Skipping — 870/870 files already present
Processing 2021/scc20210105.html.gz...
  Skipping — 811/811 files already present
Processing 2021/scc20210106.html.gz...
  Skipping — 812/812 files already present
Processing 2021/scc20210107.html.gz...
  Skipping — 803/803 files already present
Processing 2021/scc20210108.html.gz...
  Skipping — 820/820 files already present
Processing 2021/scc20210109.html.gz...
  Skipping — 1040/1040 files already present
Processing 2021/scc20210110.html.gz...
  Skipping — 1237/1237 files already present
Processing 2021/scc20210111.html.gz...
  Skipping — 1106/1106 files already present
Processing 2021/scc20210112.html.gz...
  Skipping — 821/821 files already present
Processi